In [2]:
import numpy as np
import pandas as pd

In [ ]:
data = pd.read_csv("./data/survival_pdds.txt", sep="\t")
data['date'] = pd.to_datetime(data['date'], format='%m/%d/%y')
data['sampleDate'] = pd.to_datetime(data['sampleDate'], format='%m/%d/%y')
data['min_date'] = data.groupby('id_participant')['sampleDate'].transform('min')
data['days_from_start'] = (data['date'] - data['min_date']).dt.days
data_filtered = data[['id_participant', 'score', 'days_from_start']]
data_filtered = data_filtered[data_filtered['days_from_start'] >= 0]
output_file = "./data/survival_pdds_days.tsv"
data_filtered.to_csv(output_file, sep="\t", index=False)

print(f"File saved to: {output_file}")

In [4]:
file_path = "./data/names_transfer.txt"
df = pd.read_csv(file_path, sep='\t', header=None)
names_dict = dict(zip(df[0], df[1]))

column_names_path = "./data/names_order.pdds_severe_vs_mild_with_totals.txt"
with open(column_names_path, 'r') as f:
    column_names = f.read().strip().split('\n')

bed_file_path = "./dmrs/dmrs.pdds_severe_vs_mild.add_value.methy.bed"
df = pd.read_csv(bed_file_path, sep='\t', names=column_names)
df.drop(columns=['chrom', 'start', 'end', 'blank1', 'blank2'], inplace=True)

for col in df.columns:
    if not col.endswith("_Total") and col!="key": 
        total_col = f"{col}_Total"
        if total_col in df.columns:
            df[f"{col}_normalized"] = np.where(
                (df[col] == 0) & (df[total_col] == 0),
                np.nan,
                df[col] / df[total_col]
            )
df = df.loc[:, ['key'] + [col for col in df.columns if col.endswith('_normalized')]]
df.columns = ['key'] + [names_dict[col.split('.')[0]] for col in df.columns[1:]]


In [5]:
output_file_path = "./dmr_data.txt"
df.to_csv(output_file_path, sep='\t', index=False)